# Task 5 – Model Evaluation & Stability
**Focus:** Metrics, ROC/PR Curves, Perturbation Analysis, SHAP/LIME


In [1]:
# ============================================================
# TASK 5: Model Evaluation & Stability
# ============================================================

from pyspark.sql import SparkSession
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col, when, rand
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

spark = SparkSession.builder \
    .appName('7006SCN_Task5_Evaluation') \
    .config('spark.executor.memory', '4g') \
    .getOrCreate()

print('Task 5: Model Evaluation & Stability')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/29 15:53:54 WARN Utils: Your hostname, Naveenrajs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.34 instead (on interface en0)
26/06/29 15:53:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 15:53:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/29 15:53:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/29 15:53:54 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Task 5: Model Evaluation & Stability


In [2]:
# ============================================================
# LOAD PREDICTIONS FROM TASK 3
# ============================================================

lr_preds = spark.read.parquet('./data/lr_predictions.parquet')
rf_preds = spark.read.parquet('./data/rf_predictions.parquet')
gbt_preds = spark.read.parquet('./data/gbt_predictions.parquet')
svm_preds = spark.read.parquet('./data/svm_predictions.parquet')

models = {
    'Logistic Regression': lr_preds,
    'Random Forest': rf_preds,
    'GBT': gbt_preds,
    'Linear SVM': svm_preds
}

print('Predictions loaded for all 4 models')

Predictions loaded for all 4 models


In [3]:
# ============================================================
# METRICS COMPARISON TABLE
# ============================================================

binary_eval = BinaryClassificationEvaluator(labelCol='high_tip')
multi_eval = MulticlassClassificationEvaluator(labelCol='high_tip', predictionCol='prediction')

all_metrics = []
for name, preds in models.items():
    auc = binary_eval.evaluate(preds)
    acc = multi_eval.evaluate(preds, {multi_eval.metricName: 'accuracy'})
    prec = multi_eval.evaluate(preds, {multi_eval.metricName: 'weightedPrecision'})
    rec = multi_eval.evaluate(preds, {multi_eval.metricName: 'weightedRecall'})
    f1 = multi_eval.evaluate(preds, {multi_eval.metricName: 'f1'})
    all_metrics.append({
        'Model': name, 'AUC-ROC': round(auc,4),
        'Accuracy': round(acc,4), 'Precision': round(prec,4),
        'Recall': round(rec,4), 'F1-Score': round(f1,4)
    })

metrics_df = pd.DataFrame(all_metrics).set_index('Model')
print('=== FULL METRICS COMPARISON TABLE ===')
print(metrics_df.to_string())

=== FULL METRICS COMPARISON TABLE ===
                     AUC-ROC  Accuracy  Precision  Recall  F1-Score
Model                                                              
Logistic Regression   0.7840    0.7957     0.8457  0.7957    0.7741
Random Forest         0.7909    0.7978     0.8475  0.7978    0.7767
GBT                   0.7920    0.7977     0.8465  0.7977    0.7768
Linear SVM            0.7530    0.7957     0.8468  0.7957    0.7739


In [4]:
# ============================================================
# CONFUSION MATRICES
# ============================================================

from pyspark.mllib.evaluation import MulticlassMetrics

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
model_list = list(models.items())

for idx, (name, preds) in enumerate(model_list):
    ax = axes[idx // 2][idx % 2]

    pred_and_labels = preds.select('prediction', col('high_tip').cast('double'))
    metrics = MulticlassMetrics(pred_and_labels.rdd.map(tuple))
    cm = metrics.confusionMatrix().toArray()

    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.set_title(f'Confusion Matrix - {name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Low Tip', 'High Tip'])
    ax.set_yticklabels(['Low Tip', 'High Tip'])

    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('./outputs/confusion_matrices.png', dpi=150, bbox_inches='tight')
print('Confusion matrices saved to ./outputs/confusion_matrices.png')
plt.show()

/Users/naveenraj/Library/Python/3.9/lib/python/site-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(
/Users/naveenraj/Library/Python/3.9/lib/python/site-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(
/Users/naveenraj/Library/Python/3.9/lib/python/site-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(
/Users/naveenraj/Library/Python/3.9/lib/python/site-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Confusion matrices saved to ./outputs/confusion_matrices.png


/var/folders/cj/nlygf9jx419395gx_zxm71_r0000gn/T/ipykernel_70221/1035848866.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ============================================================
# ROC CURVES
# ============================================================

import os
os.makedirs("./outputs", exist_ok=True)

from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(10, 8))
colors = ["blue", "green", "red", "purple"]

for (name, preds), color in zip(models.items(), colors):

    # Skip models without probability (Linear SVM)
    if "probability" not in preds.columns:
        print(f"Skipping {name} (no probability column)")
        continue

    # Sample for ROC computation
    sample = preds.select("probability", "high_tip").sample(0.1, seed=42).toPandas()

    scores = sample["probability"].apply(lambda x: float(x[1]))
    labels = sample["high_tip"]

    fpr, tpr, _ = roc_curve(labels, scores)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, lw=2, color=color,
             label=f"{name} (AUC={roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves – All Models")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)

plt.savefig("./outputs/roc_curves.png", dpi=150, bbox_inches="tight")
print("ROC curves saved to ./outputs/roc_curves.png")

plt.show()

Skipping Linear SVM (no probability column)
ROC curves saved to ./outputs/roc_curves.png


/var/folders/cj/nlygf9jx419395gx_zxm71_r0000gn/T/ipykernel_70221/2792065144.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ============================================================
# PRECISION-RECALL CURVES
# ============================================================

from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure(figsize=(10, 8))

for (name, preds), color in zip(models.items(), colors):

    # Skip models without probability (Linear SVM)
    if "probability" not in preds.columns:
        print(f"Skipping {name} (no probability column)")
        continue

    # Sample data
    sample = preds.select("probability", "high_tip").sample(0.1, seed=42).toPandas()

    scores = sample["probability"].apply(lambda x: float(x[1]))
    labels = sample["high_tip"]

    precision, recall, _ = precision_recall_curve(labels, scores)
    ap = average_precision_score(labels, scores)

    plt.plot(
        recall,
        precision,
        color=color,
        lw=2,
        label=f"{name} (AP={ap:.3f})"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves – All Models")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)

plt.savefig("./outputs/pr_curves.png", dpi=150, bbox_inches="tight")
print("PR curves saved to ./outputs/pr_curves.png")

plt.show()

Skipping Linear SVM (no probability column)
PR curves saved to ./outputs/pr_curves.png


/var/folders/cj/nlygf9jx419395gx_zxm71_r0000gn/T/ipykernel_70221/1542494029.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ============================================================
# STABILITY ANALYSIS – PERTURBATION TEST
# Inject noise into features and measure accuracy drop
# ============================================================

from pyspark.sql.functions import randn

test_df = spark.read.parquet('./data/test_processed.parquet')

def add_noise(df, noise_level=0.1):
    """Add Gaussian noise to features vector via column-level perturbation"""
    from pyspark.ml.feature import VectorAssembler
    from pyspark.sql.functions import udf
    from pyspark.ml.linalg import Vectors, VectorUDT
    import numpy as np

    @udf(returnType=VectorUDT())
    def perturb_vector(v):
        arr = np.array(v.toArray())
        noise = np.random.normal(0, noise_level * arr.std(), arr.shape)
        return Vectors.dense(arr + noise)

    return df.withColumn('features', perturb_vector(col('features')))

noise_levels = [0.0, 0.05, 0.10, 0.20]
stability_results = []

# Test with Logistic Regression and Random Forest (fastest)
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

for noise in noise_levels:
    perturbed = add_noise(test_df, noise) if noise > 0 else test_df
    for name, preds in [('LR', lr_preds), ('RF', rf_preds)]:
        acc = multi_eval.evaluate(preds, {multi_eval.metricName: 'accuracy'})
        stability_results.append({'Noise': noise, 'Model': name, 'Accuracy': round(acc, 4)})

stab_df = pd.DataFrame(stability_results)
print('=== STABILITY ANALYSIS – PERTURBATION RESULTS ===')
print(stab_df.pivot(index='Noise', columns='Model', values='Accuracy').to_string())

=== STABILITY ANALYSIS – PERTURBATION RESULTS ===
Model      LR      RF
Noise                
0.00   0.7957  0.7978
0.05   0.7957  0.7978
0.10   0.7957  0.7978
0.20   0.7957  0.7978


In [12]:
# ============================================================
# MODEL EXPLAINABILITY – SHAP (via sklearn bridge)
# ============================================================

# Sample a small subset for SHAP (computationally intensive)
sample_pdf = test_df.select('features', 'high_tip').sample(0.001, seed=42).toPandas()

# Convert Spark vectors to numpy
X_sample = np.array(sample_pdf['features'].tolist())
y_sample = sample_pdf['high_tip'].values

# Train a sklearn RF on sample for SHAP
from sklearn.ensemble import RandomForestClassifier as SklearnRF
import shap

train_sample = train_df.select('features', 'high_tip').sample(0.001, seed=42).toPandas()
X_train_s = np.array(train_sample['features'].tolist())
y_train_s = train_sample['high_tip'].values

feature_names = [
    'trip_distance', 'fare_amount', 'pickup_hour',
    'pickup_day', 'pickup_month', 'is_weekend',
    'is_rush_hour', 'trip_duration_min', 'speed_mph',
    'fare_per_mile', 'passenger_count',
    'payment_type_idx', 'vendor_idx'
]

rf_sklearn = SklearnRF(n_estimators=50, random_state=42)
rf_sklearn.fit(X_train_s, y_train_s)

explainer = shap.TreeExplainer(rf_sklearn)
shap_values = explainer.shap_values(X_sample)

plt.figure(figsize=(10, 6))

# Compatible with both old and new SHAP versions
if isinstance(shap_values, list):
    shap.summary_plot(
        shap_values[1],
        X_sample,
        feature_names=feature_names,
        show=False
    )
else:
    shap.summary_plot(
        shap_values,
        X_sample,
        feature_names=feature_names,
        show=False
    )
plt.title('SHAP Feature Importance – Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./outputs/shap_summary.png', dpi=150, bbox_inches='tight')
print('SHAP summary plot saved to ./outputs/shap_summary.png')
plt.show()

# spark.stop()

SHAP summary plot saved to ./outputs/shap_summary.png


/var/folders/cj/nlygf9jx419395gx_zxm71_r0000gn/T/ipykernel_70221/1382312227.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
